# H3L1 — Beyond the training length

**Week 6, notebook 3.** Attention does not see the order of its inputs: the
two tokens "the" at positions 0 and 4 enter as the same vector. Position has
to be added, and there are two main ways to add it:

| | how | used by |
|---|---|---|
| **learned absolute positions** | a table with one trained vector per position, added to the input | GPT-2, BERT |
| **RoPE** | each query and key rotated by an angle proportional to its position | Llama, Qwen and most current models |

Within the training length both work. This notebook shows what happens
**beyond it**, where the two designs stop being interchangeable.

We train the same small transformer twice, once with each scheme, on a copy
task at length **64**, then give both models longer sequences and compare.

**Colab:** Runtime → Run all. About 40 seconds on a GPU, about 3 minutes on the
CPU runtime.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0); np.random.seed(0)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV)

C_BLUE, C_RED, C_TEAL, C_PURPLE, C_GRAY = \
    "#1F4E79", "#C00000", "#0E7C7B", "#7030A0", "#595959"

# ---------------------------------------------------------------- KNOB
L_TRAIN = 64            # every training sequence has exactly this length
# ----------------------------------------------------------------------

V, D_MODEL, H = 64, 64, 4
D_HEAD = D_MODEL // H
BOS, SEP, PAD = 1, 2, 0
MAX_M_TRAIN = (L_TRAIN - 2) // 2          # longest trainable segment: 31
STEPS, BATCH = 600, 64

def make_batch(n, m=None, T=None):
    """The copy task: a segment, a separator, the segment again.
    At evaluation, larger m (with larger T) pushes past the trained length."""
    ms = torch.full((n,), m, dtype=torch.long) if m else \
        torch.randint(2, MAX_M_TRAIN + 1, (n,))
    T = T or L_TRAIN
    x = torch.full((n, T), PAD, dtype=torch.long)
    mask = torch.zeros(n, T - 1, dtype=torch.bool)
    x[:, 0] = BOS
    for i in range(n):
        mi = int(ms[i])
        seg = torch.randint(4, V, (mi,))
        x[i, 1:1 + mi] = seg
        x[i, 1 + mi] = SEP
        x[i, 2 + mi:2 + 2 * mi] = seg
        mask[i, 1 + mi:1 + 2 * mi] = True
    return x.to(DEV), mask.to(DEV)


## The two position schemes, side by side in code

**Learned absolute** is an embedding table indexed by position — nothing
more. Its whole contract is visible in the signature: `pos_table[position]`.
A position the table has no row for *does not exist*.

**RoPE** is a function, not a table. Adjacent number pairs in $q$ and $k$
are read as 2-D points and rotated by $m\,\theta_j$ — angle proportional
to the token's position $m$, at a per-pair rate $\theta_j$. The score
between a query at $m$ and a key at $n$ then depends only on the **offset**
$n-m$ (H1L1 takes this apart). Any real number is a legal position —
whether the *trained weights* can use an unseen one is exactly what we
are about to measure.


In [ ]:
def rope_rotate(x, positions):
    """x [B, H, T, d_head], positions [T] (float ok -- PI needs fractions)."""
    j = torch.arange(D_HEAD // 2, device=x.device, dtype=torch.float32)
    theta = 10000.0 ** (-2.0 * j / D_HEAD)
    ang = positions.float()[:, None] * theta[None, :]        # [T, d/2]
    c, s = ang.cos()[None, None], ang.sin()[None, None]
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.empty_like(x)
    out[..., 0::2] = x1 * c - x2 * s
    out[..., 1::2] = x1 * s + x2 * c
    return out

class Block(nn.Module):
    def __init__(self, use_rope):
        super().__init__()
        self.use_rope = use_rope
        self.ln1, self.ln2 = nn.LayerNorm(D_MODEL), nn.LayerNorm(D_MODEL)
        self.wq, self.wk, self.wv, self.wo = (nn.Linear(D_MODEL, D_MODEL, bias=False)
                                              for _ in range(4))
        self.mlp = nn.Sequential(nn.Linear(D_MODEL, 4 * D_MODEL), nn.GELU(),
                                 nn.Linear(4 * D_MODEL, D_MODEL))
        self.last_attn = None
    def forward(self, x, positions):
        Bn, Tn, _ = x.shape
        h = self.ln1(x)
        split = lambda t: t.view(Bn, Tn, H, D_HEAD).transpose(1, 2)
        q, k, v = split(self.wq(h)), split(self.wk(h)), split(self.wv(h))
        if self.use_rope:
            q, k = rope_rotate(q, positions), rope_rotate(k, positions)
        s = q @ k.transpose(-1, -2) / math.sqrt(D_HEAD)
        s = s + torch.triu(torch.full((Tn, Tn), -torch.inf, device=x.device), 1)
        p = F.softmax(s, -1)
        self.last_attn = p.detach()
        x = x + self.wo((p @ v).transpose(1, 2).reshape(Bn, Tn, D_MODEL))
        return x + self.mlp(self.ln2(x))

class TinyLM(nn.Module):
    def __init__(self, pos_mode, n_pos=L_TRAIN):
        super().__init__()
        self.pos_mode = pos_mode
        self.emb = nn.Embedding(V, D_MODEL)
        if pos_mode == "learned":
            self.pos_table = nn.Embedding(n_pos, D_MODEL)    # THE table
        self.blocks = nn.ModuleList([Block(pos_mode == "rope") for _ in range(2)])
        self.head = nn.Linear(D_MODEL, V)
    def forward(self, x, pos_scale=1.0):
        Tn = x.shape[1]
        positions = torch.arange(Tn, device=x.device) / pos_scale
        h = self.emb(x)
        if self.pos_mode == "learned":
            h = h + self.pos_table(positions.long())
        for b in self.blocks:
            h = b(h, positions)
        return self.head(h)

def train(pos_mode, n_pos=L_TRAIN, seed=1):
    torch.manual_seed(seed)
    m = TinyLM(pos_mode, n_pos).to(DEV)
    opt = torch.optim.Adam(m.parameters(), lr=2e-3)
    for step in range(STEPS):
        x, mask = make_batch(BATCH)
        logits = m(x)[:, :-1]
        tgt = x[:, 1:].clone(); tgt[~mask] = -100
        loss = F.cross_entropy(logits.reshape(-1, V), tgt.reshape(-1),
                               ignore_index=-100)
        opt.zero_grad(); loss.backward(); opt.step()
    return m, loss.item()

def accuracy(model, m, pos_scale=1.0, n=512):
    T = 2 + 2 * m
    with torch.no_grad():
        x, mask = make_batch(n, m=m, T=T)
        pred = model(x, pos_scale=pos_scale)[:, :-1].argmax(-1)
        return (((pred == x[:, 1:]) & mask).sum() / mask.sum()).item()


## First difference: position 64 *does not exist* for the table

Before training anything, the cheapest experiment in the notebook. An
honest implementation sizes the table to the training length — GPT-2's has
1024 rows because 1024 was its context. Hand such a model a sequence of
length 92 and it does the only thing a table can do:


In [ ]:
x_long, _ = make_batch(4, m=45, T=92)
for mode in ("rope", "learned"):
    try:
        with torch.no_grad():
            TinyLM(mode, n_pos=L_TRAIN).cpu()(x_long.cpu())
        print(f"{mode:8s}: ran fine at length 92")
    except IndexError as e:
        print(f"{mode:8s}: {type(e).__name__} -- {e}")
        print("          it does not degrade past its table. It STOPS.")


No training was needed for that, which is the point — it is a property of
the lookup, not of the weights. (On CPU, deliberately: on a GPU that same
out-of-range lookup does not raise a polite `IndexError`, it trips a
**device-side assert that poisons the whole CUDA context**. The production
version of this crash is worse than the one above.)

## Train both at length 64 — and only ever at length 64

Nobody ships the crash, of course: they allocate a bigger table "for
headroom". So we give the absolute model **256 rows** and train on only the
first 64. The rest exist, hold their random initialization, and never
receive a gradient — which makes this the honest comparison,
**untrained rows vs unseen angles.**


In [ ]:
models = {}
models["rope"], loss_r = train("rope")
models["learned"], loss_l = train("learned", n_pos=256)
for mode, ls in (("rope", loss_r), ("learned", loss_l)):
    print(f"{mode:8s} trained: final loss {ls:.3f}, copy accuracy at "
          f"distance 24 (in-range): {accuracy(models[mode], 24):.2f}")


## The measurement: accuracy vs length

Copy accuracy as the copy distance $m$ (and with it the sequence length
$2m{+}2$) grows past everything the models trained on. Three curves:

* **learned absolute** (256 rows, 64 trained),
* **RoPE**, positions used as-is (naive extrapolation),
* **RoPE + position interpolation (PI)**: same weights, but positions are
  **divided by** $T/64$ at evaluation, squeezing every angle back into the
  trained range. One line of code, no retraining — the first move of the
  whole context-extension literature (deck §2.5).


In [ ]:
eval_ms = [12, 24, 31, 45, 60, 90, 120]
eval_lens = [2 + 2 * m for m in eval_ms]
curves = {"learned absolute": [], "RoPE (naive)": [], "RoPE + PI": []}
for m in eval_ms:
    T = 2 + 2 * m
    curves["learned absolute"].append(accuracy(models["learned"], m))
    curves["RoPE (naive)"].append(accuracy(models["rope"], m))
    curves["RoPE + PI"].append(
        accuracy(models["rope"], m, pos_scale=max(1.0, T / L_TRAIN)))

fig, ax = plt.subplots(figsize=(7.8, 4.2))
ax.axvspan(0, L_TRAIN, color=C_GRAY, alpha=0.10, lw=0)
ax.text(L_TRAIN * 0.5, 0.06, "trained range", ha="center", fontsize=9, color=C_GRAY)
for (name, a), c, mk in zip(curves.items(), [C_RED, C_BLUE, C_TEAL], "sD^"):
    ax.plot(eval_lens, a, mk + "-", color=c, label=name, ms=6)
ax.axhline(1 / (V - 4), color=C_GRAY, ls=":", lw=1)
ax.set_xlabel("evaluation sequence length")
ax.set_ylabel("copy accuracy"), ax.set_ylim(-0.04, 1.06)
ax.set_title(f"trained at {L_TRAIN}, evaluated past the edge")
ax.legend(frameon=False, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

for name, a in curves.items():
    print(f"{name:18s} " + "  ".join(f"L={L}:{v:.2f}"
                                     for L, v in zip(eval_lens, a)))


Read the three curves against the shaded edge:

* inside it, everyone is perfect — the schemes are indistinguishable
  where training covered them;
* the **table** falls off a cliff. Its rows past 64 still hold their random
  initialization — no gradient ever reached them — so what it adds to every
  token out there is literally noise;
* **naive RoPE** degrades too — erratically: some lengths half-work, some
  collapse. Its slow pairs are being evaluated at angles they never saw —
  the 18-of-64 that H1L1 counts, now failing at a task;
* **PI** buys real accuracy at moderate extension *with zero retraining*
  — and fades at large stretch, where compressing neighbours together
  costs the fast pairs their resolution. (The production recipe adds a
  brief fine-tune at the squeezed positions — cheap precisely because
  every angle is back in distribution. That, plus the per-pair
  refinements NTK and YaRN, is deck §2.5–2.8.)

The asymmetry to keep: **you cannot interpolate a table, but you can
re-sample a function.** A table has nothing between or beyond its rows; a
function can be evaluated anywhere, and PI's one-liner squeezes every
position back inside the arc the weights were trained on. That is why the
extension literature is a RoPE literature.


## What H3L1 showed

| scheme | at $L_{train}$ | past it | the failure is |
|---|---|---|---|
| learned absolute | perfect | `IndexError`, or noise rows if over-allocated | **a missing row** — nothing exists between or beyond the entries |
| RoPE, naive | perfect | erratic decay as slow pairs leave their trained arc | **an unseen angle** — but a *continuous* one |
| RoPE + PI | — | real recovery at moderate stretch, zero retraining | positions *re-sampled*, not looked up |

**Summary.** A learned table fails at the first position beyond it; RoPE
degrades gradually, and position interpolation recovers part of the accuracy
without retraining. What generation costs at these lengths is the topic of
the KV cache.
